
# LIVE / NIMHANS Dataset EDA — VGST1 Visual-Reconstruction Workflow

This notebook provides the EDA/QC stage for the LIVE/NIMHANS fMRI dataset used in the
VGST1 visual-reconstruction workflow.

It follows the same EDA principles used for the Haxby and NSD notebooks:

**dataset inventory → BOLD QC → brain mask → temporal QC → stimulus QC → semantic/COCO QC →
fMRI feature QC → CLIP/DINOv2 feature QC → subject/shared-alignment QC → retrieval baseline QC →
final integrity report**

### Dataset-specific information used by this notebook

The current LIVE dataset organization is expected to contain subject BOLD NIfTI files such as:

`subjects/subject1_bold.nii.gz`

The known Subject 1 acquisition reported for this project is:

- spatial shape: **147 × 144 × 36**
- timepoints: **165**
- voxel size: approximately **1.796875 × 1.796875 × 2.999958 mm**
- TR: **3.0 s**
- stimulus data: natural-scene images
- semantic information: COCO-style annotations/index information where available

The notebook reads the actual NIfTI header rather than hard-coding these values for analysis.

> **Scientific distinction:** raw/mean BOLD intensity maps are QC visualizations. They are not
> statistical activation maps. A GLM/statistical activation analysis is included only as an
> optional section when valid stimulus/event timing and a suitable design matrix are available.

### VGST1-relevant EDA

The notebook prepares evidence for:

- fMRI preprocessing and brain masking
- shared/cross-subject representation alignment
- ridge baseline
- CLIP and DINOv2 visual feature spaces
- retrieval evaluation
- semantic correspondence
- downstream diffusion-reconstruction readiness

It does **not** claim model performance or reproduce training results.


In [ ]:

# ============================================================
# 1. CONFIGURATION
# ============================================================

from pathlib import Path
import os
import glob
import json
import gc
import random
import warnings

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Dataset root
# ------------------------------------------------------------
# Example:
# BASE_DIR = Path(r"D:/mosaic_output")
BASE_DIR = Path(".")

SUBJECTS_DIR = BASE_DIR / "subjects"

# Optional stimulus directory
# Example:
# STIMULUS_DIR = BASE_DIR / "stimuli"
STIMULUS_DIR = None

# Optional annotation directory
# Example:
# ANNOTATION_DIR = BASE_DIR / "annotations"
ANNOTATION_DIR = None

# ------------------------------------------------------------
# Subjects
# ------------------------------------------------------------
SUBJECT_IDS = [1, 2, 3, 4, 5, 6]

# ------------------------------------------------------------
# Expected acquisition values are QC references only.
# Actual values are always read from each NIfTI header.
# ------------------------------------------------------------
EXPECTED_TIMEPOINTS = 165
EXPECTED_TR = 3.0
EXPECTED_SPATIAL_SHAPE = (147, 144, 36)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------
EDA_DIR = BASE_DIR / "live_eda_outputs"
FIG_DIR = EDA_DIR / "figures"
TABLE_DIR = EDA_DIR / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Base directory:", BASE_DIR.resolve())
print("Subjects directory:", SUBJECTS_DIR.resolve())
print("EDA directory:", EDA_DIR.resolve())


In [ ]:

# ============================================================
# 2. PACKAGE CHECK
# ============================================================

packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "nibabel": "nibabel",
    "matplotlib": "matplotlib",
    "PIL": "Pillow",
    "sklearn": "scikit-learn",
}

missing = []

for module_name, package_name in packages.items():
    try:
        __import__(module_name)
    except Exception:
        missing.append(package_name)

if missing:
    print("Missing packages:", sorted(set(missing)))
    print("%pip install numpy pandas nibabel matplotlib pillow scikit-learn")
else:
    print("All base EDA packages are available.")



## 3. Dataset inventory

Checks the six LIVE subjects, BOLD NIfTI files, optional masks, stimulus files and optional
annotation files.


In [ ]:

# ============================================================
# 3. DATASET INVENTORY
# ============================================================

def find_bold(subject_id):
    candidates = [
        SUBJECTS_DIR / f"subject{subject_id}_bold.nii.gz",
        SUBJECTS_DIR / f"subject{subject_id}_bold.nii",
        SUBJECTS_DIR / f"subject_{subject_id}_bold.nii.gz",
        SUBJECTS_DIR / f"subject_{subject_id}_bold.nii",
        SUBJECTS_DIR / f"sub{subject_id:02d}_bold.nii.gz",
        SUBJECTS_DIR / f"sub{subject_id:02d}_bold.nii",
    ]

    for p in candidates:
        if p.exists():
            return p

    patterns = [
        SUBJECTS_DIR / f"*subject{subject_id}*bold*.nii*",
        SUBJECTS_DIR / f"*subject_{subject_id}*bold*.nii*",
        SUBJECTS_DIR / f"*sub{subject_id:02d}*bold*.nii*",
    ]

    for pattern in patterns:
        matches = sorted(glob.glob(str(pattern)))
        if matches:
            return Path(matches[0])

    return None


def find_mask(subject_id):
    candidates = [
        SUBJECTS_DIR / f"subject{subject_id}_brain_mask.nii.gz",
        SUBJECTS_DIR / f"subject{subject_id}_brain_mask.nii",
        SUBJECTS_DIR / f"subject_{subject_id}_brain_mask.nii.gz",
        SUBJECTS_DIR / f"sub{subject_id:02d}_brain_mask.nii.gz",
    ]

    for p in candidates:
        if p.exists():
            return p

    return None


inventory_rows = []

for sid in SUBJECT_IDS:
    bold = find_bold(sid)
    mask = find_mask(sid)

    inventory_rows.append({
        "subject": sid,
        "bold_path": str(bold) if bold else None,
        "mask_path": str(mask) if mask else None,
        "bold_exists": bold is not None,
        "mask_exists": mask is not None,
    })

inventory = pd.DataFrame(inventory_rows)
display(inventory)

inventory.to_csv(
    TABLE_DIR / "live_dataset_inventory.csv",
    index=False
)



## 4. BOLD header and acquisition QC

The complete 4-D BOLD array is **not** loaded. Only NIfTI metadata are read.


In [ ]:

# ============================================================
# 4. BOLD HEADER QC
# ============================================================

header_rows = []

for sid in SUBJECT_IDS:
    path = find_bold(sid)

    if path is None:
        continue

    try:
        img = nib.load(str(path))
        shape = tuple(img.shape)
        zooms = img.header.get_zooms()

        row = {
            "subject": sid,
            "shape": shape,
            "x": shape[0] if len(shape) >= 1 else None,
            "y": shape[1] if len(shape) >= 2 else None,
            "z": shape[2] if len(shape) >= 3 else None,
            "timepoints": shape[3] if len(shape) >= 4 else None,
            "voxel_x_mm": float(zooms[0]) if len(zooms) >= 1 else None,
            "voxel_y_mm": float(zooms[1]) if len(zooms) >= 2 else None,
            "voxel_z_mm": float(zooms[2]) if len(zooms) >= 3 else None,
            "TR_s": float(zooms[3]) if len(zooms) >= 4 else None,
            "dtype": str(img.get_data_dtype()),
            "orientation": "".join(nib.aff2axcodes(img.affine)),
        }

        header_rows.append(row)
        del img

    except Exception as e:
        print(f"Subject {sid} header error:", repr(e))

header_qc = pd.DataFrame(header_rows)
display(header_qc)

header_qc.to_csv(
    TABLE_DIR / "live_bold_header_qc.csv",
    index=False
)

print("\nReference check:")
print("Expected spatial shape:", EXPECTED_SPATIAL_SHAPE)
print("Expected timepoints:", EXPECTED_TIMEPOINTS)
print("Expected TR:", EXPECTED_TR)


In [ ]:

# ============================================================
# 5. ACQUISITION CONSISTENCY CHECK
# ============================================================

qc_rows = []

for _, row in header_qc.iterrows():

    shape_ok = tuple(row["shape"][:3]) == EXPECTED_SPATIAL_SHAPE
    nt_ok = row["timepoints"] == EXPECTED_TIMEPOINTS

    tr_ok = (
        row["TR_s"] is not None
        and np.isclose(row["TR_s"], EXPECTED_TR, atol=1e-4)
    )

    qc_rows.append({
        "subject": row["subject"],
        "spatial_shape_ok": shape_ok,
        "timepoints_ok": nt_ok,
        "TR_ok": tr_ok,
        "orientation": row["orientation"],
        "overall_reference_match": bool(shape_ok and nt_ok and tr_ok),
    })

acquisition_qc = pd.DataFrame(qc_rows)
display(acquisition_qc)

acquisition_qc.to_csv(
    TABLE_DIR / "live_acquisition_consistency.csv",
    index=False
)



## 6. Brain-mask QC

Uses an existing subject-specific mask when available. If a mask is absent, the fallback mask
is generated from nonzero finite BOLD values **only for exploratory visualization**.

For scientific preprocessing, use the project's verified anatomical/brain mask rather than
treating nonzero intensity as a definitive anatomical brain segmentation.


In [ ]:

# ============================================================
# 6. BRAIN MASK QC
# ============================================================

def get_mask_for_subject(img, mask_path=None):
    spatial_shape = tuple(img.shape[:3])

    if mask_path is not None and Path(mask_path).exists():
        mask_img = nib.load(str(mask_path))
        mask = np.asarray(mask_img.dataobj) > 0

        if mask.shape != spatial_shape:
            raise ValueError(
                f"Mask shape {mask.shape} != BOLD shape {spatial_shape}"
            )

        return mask.astype(bool), "provided_mask"

    # Memory-safe fallback: inspect one volume at a time.
    mask = np.zeros(
        spatial_shape,
        dtype=bool
    )

    nt = img.shape[3]

    for t in range(nt):
        vol = np.asarray(
            img.dataobj[:, :, :, t],
            dtype=np.float32
        )

        mask |= (
            np.isfinite(vol)
            & (vol != 0)
        )

        del vol

    return mask, "nonzero_finite_fallback"


mask_rows = []

for sid in SUBJECT_IDS:

    bold_path = find_bold(sid)

    if bold_path is None:
        continue

    mask_path = find_mask(sid)

    img = nib.load(str(bold_path))

    try:
        mask, source = get_mask_for_subject(
            img,
            mask_path
        )

        n_voxels = int(mask.sum())
        total_voxels = int(np.prod(img.shape[:3]))

        mask_rows.append({
            "subject": sid,
            "mask_source": source,
            "brain_voxels": n_voxels,
            "total_spatial_voxels": total_voxels,
            "brain_fraction": n_voxels / total_voxels,
        })

        del mask

    except Exception as e:
        print(f"Subject {sid} mask error:", repr(e))

    del img
    gc.collect()

mask_qc = pd.DataFrame(mask_rows)
display(mask_qc)

mask_qc.to_csv(
    TABLE_DIR / "live_brain_mask_qc.csv",
    index=False
)



## 7. Memory-safe temporal mean and orthogonal brain maps

This section avoids loading the full 4-D BOLD dataset. It calculates only the three central
orthogonal temporal-mean slices required for QC.

Rows:

- Axial
- Coronal
- Sagittal

Columns:

- Subject 1–6


In [ ]:

# ============================================================
# 7. MEMORY-SAFE ORTHOGONAL TEMPORAL-MEAN MAPS
# ============================================================

from matplotlib.colors import LinearSegmentedColormap

brain_cmap = LinearSegmentedColormap.from_list(
    "LIVE_BOLD",
    [
        "#00145c",
        "#003caa",
        "#0877dc",
        "#75bdf5",
        "#ffffff",
        "#fff0d8",
        "#ff9b6a",
        "#ef4328",
        "#a90000",
    ]
)


def read_slice(
    dataobj,
    plane,
    index,
    timepoint
):
    if plane == "axial":
        return np.asarray(
            dataobj[:, :, index, timepoint],
            dtype=np.float32
        )

    if plane == "coronal":
        return np.asarray(
            dataobj[:, index, :, timepoint],
            dtype=np.float32
        )

    if plane == "sagittal":
        return np.asarray(
            dataobj[index, :, :, timepoint],
            dtype=np.float32
        )

    raise ValueError(plane)


def temporal_mean_slice(
    dataobj,
    plane,
    index,
    nt
):
    first = read_slice(
        dataobj,
        plane,
        index,
        0
    )

    accumulator = np.zeros(
        first.shape,
        dtype=np.float64
    )

    valid_count = np.zeros(
        first.shape,
        dtype=np.uint16
    )

    for t in range(nt):

        if t == 0:
            arr = first
        else:
            arr = read_slice(
                dataobj,
                plane,
                index,
                t
            )

        finite = np.isfinite(arr)

        accumulator[finite] += arr[finite]
        valid_count[finite] += 1

        if t != 0:
            del arr

    result = np.zeros(
        accumulator.shape,
        dtype=np.float32
    )

    valid = valid_count > 0

    result[valid] = (
        accumulator[valid]
        / valid_count[valid]
    ).astype(np.float32)

    del accumulator
    del valid_count
    gc.collect()

    return result


orthogonal_results = {}

for sid in SUBJECT_IDS:

    path = find_bold(sid)

    if path is None:
        continue

    print(
        f"Processing Subject {sid}: {path.name}"
    )

    img = nib.load(str(path))

    nx, ny, nz, nt = img.shape

    x_idx = nx // 2
    y_idx = ny // 2
    z_idx = nz // 2

    axial = temporal_mean_slice(
        img.dataobj,
        "axial",
        z_idx,
        nt
    )

    coronal = temporal_mean_slice(
        img.dataobj,
        "coronal",
        y_idx,
        nt
    )

    sagittal = temporal_mean_slice(
        img.dataobj,
        "sagittal",
        x_idx,
        nt
    )

    orthogonal_results[sid] = {
        "axial": axial,
        "coronal": coronal,
        "sagittal": sagittal,
        "shape": img.shape,
        "x_idx": x_idx,
        "y_idx": y_idx,
        "z_idx": z_idx,
    }

    del img
    gc.collect()

print(
    "Processed subjects:",
    sorted(orthogonal_results.keys())
)


In [ ]:

# ============================================================
# 8. COMMON COLOR SCALE AND 3-VIEW FIGURE
# ============================================================

all_values = []

for sid, result in orthogonal_results.items():

    for plane in ["axial", "coronal", "sagittal"]:

        image = result[plane]

        values = image[
            np.isfinite(image)
            & (image != 0)
        ]

        if values.size:
            all_values.append(values)

if not all_values:
    raise RuntimeError("No valid BOLD values.")

all_values = np.concatenate(all_values)

vmin = np.percentile(all_values, 2)
vmax = np.percentile(all_values, 98)

loaded = sorted(orthogonal_results.keys())

fig, axes = plt.subplots(
    3,
    len(loaded),
    figsize=(18, 9),
    dpi=200
)

axes = np.asarray(axes).reshape(
    3,
    len(loaded)
)

planes = ["axial", "coronal", "sagittal"]
labels = ["Axial", "Coronal", "Sagittal"]

for col, sid in enumerate(loaded):

    for row, plane in enumerate(planes):

        ax = axes[row, col]

        image = np.rot90(
            orthogonal_results[sid][plane]
        )

        image = np.ma.masked_where(
            image == 0,
            image
        )

        ax.imshow(
            image,
            cmap=brain_cmap,
            vmin=vmin,
            vmax=vmax,
            interpolation="bilinear",
            aspect="equal"
        )

        ax.set_xticks([])
        ax.set_yticks([])

        if row == 0:
            ax.set_title(
                f"Subject {sid}",
                fontsize=12,
                fontweight="bold"
            )

        if col == 0:
            ax.set_ylabel(
                labels[row],
                fontsize=11,
                fontweight="bold"
            )

        for spine in ax.spines.values():
            spine.set_linewidth(0.8)

fig.suptitle(
    "LIVE / NIMHANS — Temporal-Mean BOLD Brain Maps",
    fontsize=16,
    fontweight="bold"
)

sm = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=vmin, vmax=vmax),
    cmap=brain_cmap
)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=axes.ravel().tolist(),
    fraction=0.018,
    pad=0.015
)

cbar.set_label(
    "Mean BOLD intensity",
    fontweight="bold"
)

fig.text(
    0.5,
    0.015,
    "Temporal mean across all available BOLD timepoints; "
    "common intensity scale across subjects.",
    ha="center",
    fontsize=9
)

plt.subplots_adjust(
    left=0.05,
    right=0.94,
    top=0.90,
    bottom=0.07,
    wspace=0.03,
    hspace=0.08
)

fig.savefig(
    FIG_DIR / "live_temporal_mean_orthogonal_maps.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()



## 9. Temporal global BOLD signal

This follows the same temporal EDA used for Haxby/NSD. It computes a global mean from the
brain mask one timepoint at a time, avoiding a full 4-D memory allocation.


In [ ]:

# ============================================================
# 9. GLOBAL TEMPORAL BOLD SIGNAL
# ============================================================

def global_temporal_signal(
    img,
    mask
):
    nt = img.shape[3]
    signal = np.zeros(
        nt,
        dtype=np.float32
    )

    for t in range(nt):

        volume = np.asarray(
            img.dataobj[:, :, :, t],
            dtype=np.float32
        )

        values = volume[mask]
        values = values[np.isfinite(values)]

        if values.size:
            signal[t] = float(
                values.mean()
            )

        del volume

    return signal


temporal_signals = {}

for sid in SUBJECT_IDS:

    path = find_bold(sid)

    if path is None:
        continue

    img = nib.load(str(path))
    mask_path = find_mask(sid)

    mask, source = get_mask_for_subject(
        img,
        mask_path
    )

    signal = global_temporal_signal(
        img,
        mask
    )

    temporal_signals[sid] = {
        "signal": signal,
        "TR": (
            float(img.header.get_zooms()[3])
            if len(img.header.get_zooms()) >= 4
            else None
        ),
    }

    del mask
    del img
    gc.collect()

print(
    "Temporal signals:",
    sorted(temporal_signals.keys())
)


In [ ]:

# ============================================================
# 10. TEMPORAL SIGNAL PLOT
# ============================================================

fig, axes = plt.subplots(
    len(temporal_signals),
    1,
    figsize=(11, 2.5 * len(temporal_signals)),
    squeeze=False
)

for row, sid in enumerate(
    sorted(temporal_signals.keys())
):

    ax = axes[row, 0]

    signal = temporal_signals[sid]["signal"]
    tr = temporal_signals[sid]["TR"]

    if tr is not None:
        x = np.arange(len(signal)) * tr
        xlabel = "Time (s)"
    else:
        x = np.arange(len(signal))
        xlabel = "Timepoint"

    ax.plot(
        x,
        signal,
        linewidth=0.8
    )

    ax.set_title(
        f"Subject {sid} — Global Mean BOLD"
    )

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Mean BOLD")
    ax.grid(
        True,
        linestyle="--",
        linewidth=0.4,
        alpha=0.35
    )

plt.tight_layout()

fig.savefig(
    FIG_DIR / "live_global_temporal_bold.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()



## 11. Temporal variability and tSNR

For each voxel:

\[
tSNR(v) = \frac{\mu_v}{\sigma_v}
\]

This section computes tSNR using one 3-D volume at a time. The complete 4-D dataset is never
held in memory.


In [ ]:

# ============================================================
# 11. MEMORY-SAFE tSNR SUMMARY
# ============================================================

def temporal_mean_std_tsnr(
    img,
    mask
):
    nx, ny, nz, nt = img.shape

    mean = np.zeros(
        (nx, ny, nz),
        dtype=np.float64
    )

    m2 = np.zeros(
        (nx, ny, nz),
        dtype=np.float64
    )

    count = np.zeros(
        (nx, ny, nz),
        dtype=np.uint16
    )

    for t in range(nt):

        vol = np.asarray(
            img.dataobj[:, :, :, t],
            dtype=np.float32
        )

        finite = (
            mask
            & np.isfinite(vol)
        )

        count[finite] += 1

        delta = np.zeros_like(
            mean,
            dtype=np.float64
        )

        delta[finite] = (
            vol[finite]
            - mean[finite]
        )

        mean[finite] += (
            delta[finite]
            / count[finite]
        )

        delta2 = np.zeros_like(
            mean,
            dtype=np.float64
        )

        delta2[finite] = (
            vol[finite]
            - mean[finite]
        )

        m2[finite] += (
            delta[finite]
            * delta2[finite]
        )

        del vol
        del finite
        del delta
        del delta2

    std = np.zeros_like(
        mean,
        dtype=np.float64
    )

    valid = (
        mask
        & (count > 1)
    )

    std[valid] = np.sqrt(
        m2[valid]
        / (count[valid] - 1)
    )

    tsnr = np.zeros_like(
        mean,
        dtype=np.float32
    )

    valid_tsnr = (
        valid
        & (std > 1e-8)
    )

    tsnr[valid_tsnr] = (
        mean[valid_tsnr]
        / std[valid_tsnr]
    ).astype(np.float32)

    return (
        mean.astype(np.float32),
        std.astype(np.float32),
        tsnr
    )


tsnr_summary = []

for sid in SUBJECT_IDS:

    path = find_bold(sid)

    if path is None:
        continue

    img = nib.load(str(path))
    mask_path = find_mask(sid)

    mask, source = get_mask_for_subject(
        img,
        mask_path
    )

    mean, std, tsnr = temporal_mean_std_tsnr(
        img,
        mask
    )

    values = tsnr[
        mask
        & np.isfinite(tsnr)
        & (tsnr > 0)
    ]

    tsnr_summary.append({
        "subject": sid,
        "mask_source": source,
        "median_tSNR": float(np.median(values)) if values.size else np.nan,
        "mean_tSNR": float(np.mean(values)) if values.size else np.nan,
        "p10_tSNR": float(np.percentile(values, 10)) if values.size else np.nan,
        "p90_tSNR": float(np.percentile(values, 90)) if values.size else np.nan,
    })

    del mean
    del std
    del tsnr
    del mask
    del img
    gc.collect()

tsnr_summary = pd.DataFrame(tsnr_summary)
display(tsnr_summary)

tsnr_summary.to_csv(
    TABLE_DIR / "live_tsnr_summary.csv",
    index=False
)



## 12. Stimulus inventory and display

The LIVE workflow uses natural-scene stimuli. This section checks the actual local images rather
than assuming a particular naming convention.


In [ ]:

# ============================================================
# 12. STIMULUS INVENTORY
# ============================================================

from PIL import Image

if STIMULUS_DIR is None:

    print(
        "STIMULUS_DIR is None. "
        "Set it in the configuration cell to inspect images."
    )

else:

    STIMULUS_DIR = Path(STIMULUS_DIR)

    image_extensions = {
        ".jpg", ".jpeg", ".png",
        ".webp", ".bmp", ".tif", ".tiff"
    }

    stimulus_paths = [
        p for p in STIMULUS_DIR.rglob("*")
        if p.is_file()
        and p.suffix.lower() in image_extensions
    ]

    stimulus_rows = []

    for p in stimulus_paths:

        try:
            with Image.open(p) as im:

                stimulus_rows.append({
                    "filename": p.name,
                    "path": str(p),
                    "width": im.width,
                    "height": im.height,
                    "mode": im.mode,
                    "format": im.format,
                    "size_bytes": p.stat().st_size,
                })

        except Exception as e:

            stimulus_rows.append({
                "filename": p.name,
                "path": str(p),
                "width": None,
                "height": None,
                "mode": None,
                "format": None,
                "size_bytes": p.stat().st_size,
                "error": repr(e),
            })

    stimulus_inventory = pd.DataFrame(
        stimulus_rows
    )

    print(
        "Number of images:",
        len(stimulus_inventory)
    )

    display(
        stimulus_inventory.head(20)
    )

    stimulus_inventory.to_csv(
        TABLE_DIR / "live_stimulus_inventory.csv",
        index=False
    )


In [ ]:

# ============================================================
# 13. STIMULUS SAMPLE GRID
# ============================================================

if STIMULUS_DIR is None:
    print("Skipped.")
elif len(stimulus_paths) == 0:
    print("No stimulus images found.")
else:

    rng = np.random.default_rng(
        SEED
    )

    n_show = min(
        12,
        len(stimulus_paths)
    )

    selected = list(
        rng.choice(
            stimulus_paths,
            size=n_show,
            replace=False
        )
    )

    cols = 4
    rows = int(
        np.ceil(n_show / cols)
    )

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(12, 3 * rows)
    )

    axes = np.asarray(
        axes
    ).ravel()

    for ax, path in zip(
        axes,
        selected
    ):

        with Image.open(path) as im:
            ax.imshow(
                im.convert("RGB")
            )

        ax.set_title(
            path.name,
            fontsize=8
        )

        ax.axis("off")

    for ax in axes[n_show:]:
        ax.axis("off")

    fig.suptitle(
        "LIVE Natural-Scene Stimulus Samples",
        fontsize=15,
        fontweight="bold"
    )

    plt.tight_layout()

    fig.savefig(
        FIG_DIR / "live_stimulus_samples.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()



## 14. Stimulus–fMRI pairing QC

This section checks whether an explicit mapping file exists. It does **not** invent a mapping
between an arbitrary BOLD timepoint and an image. If the project provides a CSV/JSON/NPY mapping,
set its path and inspect it here.


In [ ]:

# ============================================================
# 14. STIMULUS-FMRI PAIRING CONFIGURATION
# ============================================================

# Set this to the verified project mapping file if available.
# Examples:
# PAIRING_FILE = BASE_DIR / "stimulus_mapping.csv"
# PAIRING_FILE = BASE_DIR / "annotations" / "mapping.json"

PAIRING_FILE = None

if PAIRING_FILE is None:

    print(
        "PAIRING_FILE is None."
    )

    print(
        "No stimulus-to-timepoint pairing will be fabricated."
    )

else:

    PAIRING_FILE = Path(
        PAIRING_FILE
    )

    if not PAIRING_FILE.exists():

        raise FileNotFoundError(
            PAIRING_FILE
        )

    print(
        "Pairing file:",
        PAIRING_FILE
    )


In [ ]:

# ============================================================
# 15. PAIRING FILE INSPECTION
# ============================================================

if PAIRING_FILE is None:

    print("Skipped.")

else:

    suffix = PAIRING_FILE.suffix.lower()

    if suffix == ".csv":

        pairing = pd.read_csv(
            PAIRING_FILE
        )

    elif suffix in [".json"]:

        with open(
            PAIRING_FILE,
            "r",
            encoding="utf-8"
        ) as f:
            obj = json.load(f)

        pairing = pd.DataFrame(
            obj
        )

    elif suffix == ".npy":

        obj = np.load(
            PAIRING_FILE,
            allow_pickle=True
        )

        if obj.ndim == 0:
            obj = obj.item()

        pairing = pd.DataFrame(
            obj
        )

    else:

        raise ValueError(
            f"Unsupported pairing format: {suffix}"
        )

    print(
        "Pairing shape:",
        pairing.shape
    )

    print(
        "Columns:",
        list(pairing.columns)
    )

    display(
        pairing.head(20)
    )

    pairing.to_csv(
        TABLE_DIR / "live_verified_stimulus_fmri_pairing.csv",
        index=False
    )



## 16. COCO / semantic annotation inspection

If COCO-style annotation files are available locally, point `COCO_ANNOTATION_FILE` to the actual
JSON. The notebook reports categories and object counts without assuming that every image has the
same number of objects.


In [ ]:

# ============================================================
# 16. COCO ANNOTATION CONFIGURATION
# ============================================================

COCO_ANNOTATION_FILE = None

if COCO_ANNOTATION_FILE is None:
    print(
        "COCO_ANNOTATION_FILE is None. "
        "COCO semantic analysis skipped."
    )
else:
    COCO_ANNOTATION_FILE = Path(
        COCO_ANNOTATION_FILE
    )
    print(
        "COCO file:",
        COCO_ANNOTATION_FILE
    )


In [ ]:

# ============================================================
# 17. COCO SEMANTIC SUMMARY
# ============================================================

if COCO_ANNOTATION_FILE is None:

    print("Skipped.")

else:

    with open(
        COCO_ANNOTATION_FILE,
        "r",
        encoding="utf-8"
    ) as f:
        coco = json.load(f)

    images = pd.DataFrame(
        coco.get("images", [])
    )

    annotations = pd.DataFrame(
        coco.get("annotations", [])
    )

    categories = pd.DataFrame(
        coco.get("categories", [])
    )

    print("Images:", len(images))
    print("Annotations:", len(annotations))
    print("Categories:", len(categories))

    display(
        categories.head(20)
    )

    if not annotations.empty:

        counts = (
            annotations
            .groupby("category_id")
            .size()
            .reset_index(name="annotation_count")
        )

        if not categories.empty and "id" in categories.columns:

            counts = counts.merge(
                categories[
                    ["id", "name"]
                ],
                left_on="category_id",
                right_on="id",
                how="left"
            )

            counts = counts.drop(
                columns=["id"]
            )

        counts = counts.sort_values(
            "annotation_count",
            ascending=False
        )

        display(
            counts.head(30)
        )

        counts.to_csv(
            TABLE_DIR / "live_coco_category_counts.csv",
            index=False
        )



## 18. CLIP / DINOv2 readiness QC

The VGST1 workflow uses visual feature spaces such as CLIP and DINOv2. This EDA section does not
download large pretrained models automatically. Instead it validates already-exported embedding
files when they exist.

Expected representation:

```text
stimulus image
      ↓
CLIP embedding
      +
DINOv2 embedding
      ↓
distribution / norm / dimensionality QC
```


In [ ]:

# ============================================================
# 18. EMBEDDING FILE CONFIGURATION
# ============================================================

CLIP_EMBEDDING_FILE = None
DINOV2_EMBEDDING_FILE = None

def inspect_embedding_file(path, name):
    if path is None:
        print(f"{name}: not supplied.")
        return None

    path = Path(path)

    if not path.exists():
        print(f"{name}: file not found:", path)
        return None

    if path.suffix == ".npy":
        x = np.load(path, mmap_mode="r")
    elif path.suffix == ".npz":
        z = np.load(path)
        keys = list(z.keys())
        print(f"{name} NPZ keys:", keys)
        x = z[keys[0]]
    else:
        raise ValueError(
            f"Unsupported embedding file: {path}"
        )

    print(f"{name}")
    print(" shape:", x.shape)
    print(" dtype:", x.dtype)

    if x.ndim == 2:

        norms = np.linalg.norm(
            np.asarray(x, dtype=np.float32),
            axis=1
        )

        print(
            " mean L2 norm:",
            float(np.mean(norms))
        )

        print(
            " std L2 norm:",
            float(np.std(norms))
        )

    return x

clip_embeddings = inspect_embedding_file(
    CLIP_EMBEDDING_FILE,
    "CLIP"
)

dinov2_embeddings = inspect_embedding_file(
    DINOV2_EMBEDDING_FILE,
    "DINOv2"
)



## 19. fMRI feature readiness for shared alignment and ridge baseline

The actual VGST1 training pipeline should use the verified preprocessing implementation. This
EDA only checks whether an exported preprocessed fMRI matrix exists and reports dimensions,
missing values and basic distribution statistics.

Do not fit the final ridge baseline in this EDA notebook; keep training/evaluation splits in the
dedicated experiment notebook to prevent leakage.


In [ ]:

# ============================================================
# 19. PREPROCESSED FMRI FEATURE FILE
# ============================================================

PREPROCESSED_FMRI_FILE = None

if PREPROCESSED_FMRI_FILE is None:

    print(
        "PREPROCESSED_FMRI_FILE is None."
    )

else:

    PREPROCESSED_FMRI_FILE = Path(
        PREPROCESSED_FMRI_FILE
    )

    if PREPROCESSED_FMRI_FILE.suffix == ".npy":

        X = np.load(
            PREPROCESSED_FMRI_FILE,
            mmap_mode="r"
        )

    elif PREPROCESSED_FMRI_FILE.suffix == ".npz":

        z = np.load(
            PREPROCESSED_FMRI_FILE
        )

        print(
            "NPZ keys:",
            list(z.keys())
        )

        X = z[
            list(z.keys())[0]
        ]

    else:

        raise ValueError(
            "Use .npy or .npz for this EDA."
        )

    print(
        "Preprocessed fMRI shape:",
        X.shape
    )

    if X.ndim == 2:

        finite_fraction = np.isfinite(
            np.asarray(
                X,
                dtype=np.float32
            )
        ).mean()

        print(
            "Finite fraction:",
            float(finite_fraction)
        )



## 20. Cross-subject alignment EDA

The purpose here is descriptive: compare feature distributions before and after a verified
shared-alignment transformation.

The notebook does not implement or claim the final VGST1 alignment result. Use the dedicated
alignment module for the actual experiment.


In [ ]:

# ============================================================
# 20. ALIGNMENT OUTPUT CONFIGURATION
# ============================================================

PRE_ALIGNMENT_FILE = None
POST_ALIGNMENT_FILE = None

print(
    "Set PRE_ALIGNMENT_FILE and POST_ALIGNMENT_FILE only "
    "after generating them with the verified alignment pipeline."
)


In [ ]:

# ============================================================
# 21. ALIGNMENT DISTRIBUTION CHECK
# ============================================================

def load_matrix(path):
    path = Path(path)

    if path.suffix == ".npy":
        return np.load(
            path,
            mmap_mode="r"
        )

    if path.suffix == ".npz":
        z = np.load(path)
        return z[list(z.keys())[0]]

    raise ValueError(path)


if (
    PRE_ALIGNMENT_FILE is None
    or POST_ALIGNMENT_FILE is None
):

    print("Alignment comparison skipped.")

else:

    pre = load_matrix(
        PRE_ALIGNMENT_FILE
    )

    post = load_matrix(
        POST_ALIGNMENT_FILE
    )

    print(
        "Pre-alignment shape:",
        pre.shape
    )

    print(
        "Post-alignment shape:",
        post.shape
    )

    if pre.ndim == 2 and post.ndim == 2:

        pre_mean = np.mean(
            np.asarray(pre, dtype=np.float32),
            axis=1
        )

        post_mean = np.mean(
            np.asarray(post, dtype=np.float32),
            axis=1
        )

        plt.figure(
            figsize=(8, 5)
        )

        plt.hist(
            pre_mean,
            bins=40,
            alpha=0.55,
            label="Before alignment"
        )

        plt.hist(
            post_mean,
            bins=40,
            alpha=0.55,
            label="After alignment"
        )

        plt.xlabel(
            "Sample mean feature value"
        )

        plt.ylabel(
            "Count"
        )

        plt.title(
            "Shared-Alignment EDA"
        )

        plt.legend()
        plt.tight_layout()

        plt.savefig(
            FIG_DIR / "live_shared_alignment_eda.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()



## 22. Retrieval baseline readiness

For the VGST1 pipeline, retrieval is evaluated after the train/test split and after the fMRI-to-
feature mapping. This cell only checks that compatible fMRI and target embedding matrices can be
loaded; it does not calculate final paper metrics here.


In [ ]:

# ============================================================
# 22. RETRIEVAL INPUT CHECK
# ============================================================

RETRIEVAL_FMRI_FILE = None
RETRIEVAL_TARGET_FILE = None

if (
    RETRIEVAL_FMRI_FILE is None
    or RETRIEVAL_TARGET_FILE is None
):

    print(
        "Retrieval input check skipped. "
        "Set both verified files when available."
    )

else:

    X_retrieval = load_matrix(
        RETRIEVAL_FMRI_FILE
    )

    Y_retrieval = load_matrix(
        RETRIEVAL_TARGET_FILE
    )

    print(
        "fMRI matrix:",
        X_retrieval.shape
    )

    print(
        "Target embedding matrix:",
        Y_retrieval.shape
    )

    if X_retrieval.shape[0] != Y_retrieval.shape[0]:
        raise ValueError(
            "Sample count mismatch: "
            f"{X_retrieval.shape[0]} vs "
            f"{Y_retrieval.shape[0]}"
        )

    print(
        "Sample count is compatible for retrieval."
    )



## 23. Optional stimulus-locked / statistical activity analysis

A scientifically valid GLM/statistical activation map requires an actual event/stimulus timing
table and a valid design specification. The notebook therefore refuses to fabricate timing.

If the project has verified onset information, use the dedicated GLM notebook/module to create
beta/t/z maps. Raw BOLD percentile thresholding should not be reported as statistical activation.


In [ ]:

# ============================================================
# 23. STATISTICAL-ACTIVITY READINESS
# ============================================================

EVENT_TIMING_FILE = None

if EVENT_TIMING_FILE is None:

    print(
        "No event timing file supplied."
    )

    print(
        "GLM/statistical activation analysis is intentionally skipped."
    )

else:

    EVENT_TIMING_FILE = Path(
        EVENT_TIMING_FILE
    )

    if not EVENT_TIMING_FILE.exists():
        raise FileNotFoundError(
            EVENT_TIMING_FILE
        )

    print(
        "Verified event timing file:",
        EVENT_TIMING_FILE
    )

    print(
        "Use the project's GLM implementation to construct "
        "design matrix and statistical maps."
    )



## 24. Final integrity report

This combines the most important EDA checks into a single table suitable for repository QC.


In [ ]:

# ============================================================
# 24. FINAL INTEGRITY REPORT
# ============================================================

final_rows = []

for sid in SUBJECT_IDS:

    inv = inventory.loc[
        inventory["subject"] == sid
    ].iloc[0]

    hrow = header_qc[
        header_qc["subject"] == sid
    ]

    if len(hrow):
        hrow = hrow.iloc[0]

        shape_ok = (
            tuple(hrow["shape"][:3])
            == EXPECTED_SPATIAL_SHAPE
        )

        nt_ok = (
            hrow["timepoints"]
            == EXPECTED_TIMEPOINTS
        )

        tr_ok = (
            hrow["TR_s"] is not None
            and np.isclose(
                hrow["TR_s"],
                EXPECTED_TR,
                atol=1e-4
            )
        )
    else:
        shape_ok = nt_ok = tr_ok = False

    final_rows.append({
        "subject": sid,
        "bold_exists": bool(inv["bold_exists"]),
        "mask_exists": bool(inv["mask_exists"]),
        "header_available": len(hrow) > 0,
        "expected_spatial_shape": shape_ok,
        "expected_timepoints": nt_ok,
        "expected_TR": tr_ok,
        "temporal_signal_available": sid in temporal_signals,
        "orthogonal_maps_available": sid in orthogonal_results,
        "status": (
            "PASS"
            if (
                inv["bold_exists"]
                and shape_ok
                and nt_ok
                and tr_ok
            )
            else "CHECK"
        )
    })

final_qc = pd.DataFrame(
    final_rows
)

display(final_qc)

final_qc.to_csv(
    TABLE_DIR / "live_final_eda_qc.csv",
    index=False
)


In [ ]:

# ============================================================
# 25. FINAL JSON REPORT
# ============================================================

summary = {
    "dataset": "LIVE / NIMHANS",
    "workflow": "VGST1 visual-reconstruction EDA/QC",
    "subjects_requested": SUBJECT_IDS,
    "subjects_with_bold": [
        int(x)
        for x in inventory.loc[
            inventory["bold_exists"],
            "subject"
        ].tolist()
    ],
    "subjects_with_mask": [
        int(x)
        for x in inventory.loc[
            inventory["mask_exists"],
            "subject"
        ].tolist()
    ],
    "expected_spatial_shape": EXPECTED_SPATIAL_SHAPE,
    "expected_timepoints": EXPECTED_TIMEPOINTS,
    "expected_TR_s": EXPECTED_TR,
    "stimulus_directory": (
        str(STIMULUS_DIR)
        if STIMULUS_DIR is not None
        else None
    ),
    "eda_outputs": str(
        EDA_DIR.resolve()
    ),
}

with open(
    EDA_DIR / "live_eda_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print(
    json.dumps(
        summary,
        indent=2
    )
)

print("\nGenerated:")
print(
    "Figures:",
    FIG_DIR.resolve()
)
print(
    "Tables:",
    TABLE_DIR.resolve()
)
print(
    "Summary:",
    (EDA_DIR / "live_eda_summary.json").resolve()
)



# Final LIVE EDA checklist

Before moving to the VGST1 training/reconstruction notebooks:

- [ ] All available LIVE subjects have been inventoried.
- [ ] BOLD dimensions, voxel size, orientation and TR are recorded.
- [ ] The reported 147 × 144 × 36 × 165 / TR 3 s structure is verified from headers where applicable.
- [ ] Brain masks are checked and their source is documented.
- [ ] Mean BOLD orthogonal maps are generated.
- [ ] Global temporal BOLD curves are inspected.
- [ ] tSNR/temporal variability QC is inspected.
- [ ] Natural-scene stimulus inventory is complete when stimulus files are available.
- [ ] Stimulus-to-fMRI mapping is based only on an explicit verified mapping.
- [ ] COCO/semantic annotations are inspected when available.
- [ ] CLIP and DINOv2 embeddings are checked when exported.
- [ ] fMRI feature dimensions are compatible with the preprocessing pipeline.
- [ ] Shared-alignment inputs/outputs are checked separately from final training.
- [ ] Retrieval inputs have matching sample counts.
- [ ] GLM/statistical activation is performed only when valid event timing exists.
- [ ] No test-set information is used in preprocessing, alignment fitting, ridge fitting or retrieval-database construction.
- [ ] The final QC CSV/JSON is saved with the repository artifacts.

## Relationship to the VGST1 pipeline

```text
LIVE BOLD
   ↓
BOLD / mask / temporal QC
   ↓
preprocessing
   ↓
shared cross-subject alignment
   ↓
fMRI representation
   ├───────────────┐
   ↓               ↓
CLIP projection   DINOv2 projection
   └───────┬───────┘
           ↓
     multimodal latent
           ↓
     retrieval baseline
           ↓
      dual diffusion prior
           ↓
   Stable Diffusion reconstruction
```

This notebook is **EDA/QC only**. It does not replace the training, validation, ablation,
retrieval, or reconstruction notebooks.
